# Horizon perturbations from antenna-position error

Baseline horizon elevation profile $\alpha_h(\mathrm{az})$ (panel a) and the
change $\Delta\alpha_h = \alpha_h^{\mathrm{shift}} - \alpha_h^{\mathrm{nominal}}$
for +1 m antenna displacements East / North / Up (panel b). A 1 m move changes
the horizon by $\lesssim 0.1^\circ$ over most azimuths, spiking to $\sim 1^\circ$
only at steep cliff edges (where a lateral move slides a near-vertical horizon
edge sideways); raising the antenna (Up +1 m) lowers the horizon by a
near-uniform small offset.

Profiles are computed with `eigsep_terrain.calc_horizon` on the Marjum DEM.
Azimuth is $\mathrm{atan2}(E, N)$, North$\to$East.

Data: `horizon_perturbations.npz` (keys `names`, `az_grid`, `alpha_h`). Produces
`horizon_perturbations_1col.pdf`, the single-column figure used in the paper.
The spectral consequence of these shifts is `horizon_shift.ipynb`; the full
derivation of both is
`mock_analysis/horizon_position/notebooks/horizon_shift.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

In [ ]:
d = np.load("horizon_perturbations.npz", allow_pickle=True)
names = [str(n) for n in d["names"]]
az = np.degrees(d["az_grid"])          # azimuth grid [deg]
alpha = np.degrees(d["alpha_h"])       # horizon elevation per position [deg]
print(len(names), "positions on", az.size, "azimuths")

In [ ]:
def build_horizon_figure(az, alpha, names, out_path):
    """Single-column horizon figure: baseline profile (a), +1 m shifts (b).

    Font sizes are set per-artist rather than through ``plt.rc_context``.
    ``rc_context`` restores the ``backend`` rcParam on exit, which resets the
    inline backend's post-execute hook and silently stops every *later* cell
    from displaying its figure.
    """
    # Okabe-Ito colours, chosen to stay clear of the orange baseline fill.
    SHIFTS = [
        ("x_p_1", "East +1 m", "#0072B2"),   # blue
        ("y_p_1", "North +1 m", "#CC79A7"),  # reddish purple
        ("z_p_1", "Up +1 m", "#009E73"),     # bluish green
    ]
    FILL = "#c56a39"
    FS = 8                                   # single-column base font size
    base = alpha[names.index("nominal")]

    fig, (axt, axb) = plt.subplots(
        2, 1, figsize=(3.4, 4.0), sharex=True,
        gridspec_kw=dict(height_ratios=[3, 1.4]), layout="constrained",
    )

    axt.fill_between(az, 0, base, color=FILL, lw=0)
    axt.plot(az, base, color="black", lw=1.1)
    axt.set_ylabel("Horizon Angle [deg]", fontsize=FS)
    axt.set_ylim(0, 40)
    axt.set_axisbelow(False)   # gridlines on top of the opaque fill

    for tag, lbl, c in SHIFTS:
        axb.plot(az, alpha[names.index(tag)] - base, color=c, lw=1.0, label=lbl)
    axb.axhline(0, color="0.6", lw=0.7, ls="--")
    axb.set_ylabel(r"$\Delta$ Horizon [deg]", fontsize=FS)
    axb.set_xlabel("Azimuthal Angle [deg]", fontsize=FS)
    # Range extended past the curves to make headroom for the in-panel legend.
    axb.set_ylim(-2.1, 2.1)
    axb.legend(ncol=3, loc="upper center", fontsize=6.5, columnspacing=0.8,
               handlelength=1.2, handletextpad=0.4)

    for ax, tag in ((axt, "(a)"), (axb, "(b)")):
        ax.set_xlim(0, 360)
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=FS)
        ax.text(0.012, 0.93, tag, transform=ax.transAxes, va="top", fontsize=FS)

    fig.savefig(out_path)
    return fig

In [ ]:
fig = build_horizon_figure(az, alpha, names, "horizon_perturbations_1col.pdf")